Import libraries.

In [1]:
import os
import torch
import shutil
import kagglehub
import numpy as np
from torch import nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix

Setting of the device.

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Import dataset.

In [3]:
path = kagglehub.dataset_download("gpiosenka/sports-classification")

A generated function to extract only given classes

In [4]:
class SubsetImageFolder(datasets.ImageFolder):
    def __init__(self, root, chosen_classes, transform=None):
        self.chosen_classes = chosen_classes
        super().__init__(root, transform=transform)

    def find_classes(self, directory):
        classes = sorted(self.chosen_classes)
        class_to_idx = {}
        for i in range(len(classes)):
            class_to_idx[classes[i]] = i
        return classes, class_to_idx

Preparations of datasets. We will work with gray and then with colored data to compare the impact of color and also to reduce de number of parametters by 3 for the gray images. Our hypothesis is that for sport comparation, color will be usefull so improve accuracy.

In [5]:
IMG_SIZE = 224

transformGray = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(),
    transforms.ToTensor(),
])

transformColor = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

def create_specific_dataset(classes, transform1, transform2):
    train_datasetGray = SubsetImageFolder(os.path.join(path, "train"), classes, transform=transform1)
    val_datasetGray = SubsetImageFolder(os.path.join(path, "valid"), classes, transform=transform1)
    test_datasetGray = SubsetImageFolder(os.path.join(path, "test"), classes, transform=transform1)
    train_datasetColor = SubsetImageFolder(os.path.join(path, "train"), classes, transform=transform2)
    val_datasetColor = SubsetImageFolder(os.path.join(path, "valid"), classes, transform=transform2)
    test_datasetColor = SubsetImageFolder(os.path.join(path, "test"), classes, transform=transform2)

    return train_datasetGray, val_datasetGray, test_datasetGray, train_datasetColor, val_datasetColor, test_datasetColor

DIFF_CLASSES_3 = ["football", "nascar racing", "olympic wrestling"]
train_ds_Gray_DIFF_3, val_ds_Gray_DIFF_3, test_ds_Gray_DIFF_3, train_ds_Color_DIFF_3, val_ds_Color_DIFF_3, test_ds_Color_DIFF_3 = create_specific_dataset(DIFF_CLASSES_3, transformGray, transformColor)

SIM_CLASSES_3 = ["football", "rugby", "ampute football"]
train_ds_Gray_SIM_3, val_ds_Gray_SIM_3, test_ds_Gray_SIM_3, train_ds_Color_SIM_3, val_ds_Color_SIM_3, test_ds_Color_SIM_3 = create_specific_dataset(SIM_CLASSES_3, transformGray, transformColor)

DIFF_CLASSES_6 = ["football", "nascar racing", "olympic wrestling", "surfing", "archery", "arm wrestling"]
train_ds_Gray_DIFF_6, val_ds_Gray_DIFF_6, test_ds_Gray_DIFF_6, train_ds_Color_DIFF_6, val_ds_Color_DIFF_6, test_ds_Color_DIFF_6 = create_specific_dataset(DIFF_CLASSES_6, transformGray, transformColor)

CLASSES_12 = ["football", "olympic wrestling", "archery", "arm wrestling", "swimming", "ski jumping", "rowing", "pole dancing", "formula 1 racing", "boxing", "baton twirling", "axe throwing"]
train_ds_Gray_12, val_ds_Gray_12, test_ds_Gray_12, train_ds_Color_12, val_ds_Color_12, test_ds_Color_12 = create_specific_dataset(CLASSES_12, transformGray, transformColor)


CLASSES_20 = ["football", "nascar racing", "olympic wrestling", "surfing", "archery", "arm wrestling", "swimming", "ampute football", "skydiving", "sky surfing", "ski jumping", "rowing", "pommel horse", "pole dancing", "pole climbing", "formula 1 racing", "boxing", "baton twirling", "axe throwing", "water cycling"]
train_ds_Gray_20, val_ds_Gray_20, test_ds_Gray_20, train_ds_Color_20, val_ds_Color_20, test_ds_Color_20 = create_specific_dataset(CLASSES_20, transformGray, transformColor)

# Whole dataset ici (100 classes)
train_dsGray_100 = datasets.ImageFolder(os.path.join(path, "train"), transform=transformGray)
val_dsGray_100 = datasets.ImageFolder(os.path.join(path, "valid"), transform=transformGray)
test_dsGray_100 = datasets.ImageFolder(os.path.join(path, "test"), transform=transformGray)

train_dsColor_100 = datasets.ImageFolder(os.path.join(path, "train"), transform=transformColor)
val_dsColor_100 = datasets.ImageFolder(os.path.join(path, "valid"), transform=transformColor)
test_dsColor_100 = datasets.ImageFolder(os.path.join(path, "test"), transform=transformColor)

Here is the dataset analysis (i let you do it alvaro) I just printed number of classes and of images for each types of datasets.

In [6]:
num_classes = len(train_dsGray_100.classes)
print("Classes :", num_classes)
print("Train :", len(train_dsGray_100), "| Val :", len(val_dsGray_100), "| Test :", len(test_dsGray_100))

Classes : 100
Train : 13492 | Val : 500 | Test : 500


Convolution part

In [7]:
class Conv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, pooling):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding='same')
        self.relu = nn.ReLU()
        if pooling == "max":
            self.pool = nn.MaxPool2d(2)
        elif pooling == "average":
            self.pool = nn.AvgPool2d(2)

    def forward(self, X):
        X = self.conv(X)
        X = self.relu(X)
        X = self.pool(X)
        return X

Model Net with activation function, regularization and convolution. We set convolution by default to None, if so, it just flat the image.

In [8]:
class Net(nn.Module):
    def __init__(self, input_size, hidden_sizes=(64,), num_classes=100, convolution = None, activation_function="relu", regularization=None, reg_param=0.0):
        super().__init__()
        self.hidden_sizes = hidden_sizes
        self.num_classes = num_classes
        self.regularization = regularization
        self.reg_param = reg_param

        if convolution is not None:
            self.features = convolution
        else:
            self.features = nn.Identity()
            
        self.flatten = nn.Flatten()

        self.lst = nn.ModuleList()
        
        self.lst.append(nn.Linear(input_size, hidden_sizes[0]))
        for i in range(1, len(hidden_sizes)):
            self.lst.append(nn.Linear(hidden_sizes[i - 1], hidden_sizes[i]))
        self.lst.append(nn.Linear(hidden_sizes[-1], num_classes))

        if activation_function == "relu":
            self.activation_function = nn.ReLU()
        elif activation_function == "tanh":
            self.activation_function = nn.Tanh()
        elif activation_function == "sigmoid":
            self.activation_function = nn.Sigmoid()

        if regularization == "dropout":
            self.dropouts = nn.ModuleList()
            for i in range(len(hidden_sizes)):
                self.dropouts.append(nn.Dropout(reg_param))

    def forward(self, X):
        X = self.features(X)
        X = self.flatten(X)
        X = self.lst[0](X)
        for i in range(1, len(self.hidden_sizes) + 1):
            X = self.activation_function(X)
            if self.regularization == "dropout":
                X = self.dropouts[i - 1](X)
            X = self.lst[i](X)
        return X

Optimizers and schedulers functions.

In [9]:
def make_optimizer(name, params, lr, weight_decay=0.0):
    if name == "Adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    elif name == "SGD":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)


def make_scheduler(name, optimizer):
    if name is None:
        return None
    elif name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

Training function.

In [10]:
def train(model, train_loader, val_loader, epochs=40, lr=0.001, optimizer_name="SGD", scheduler_name=None, limit_epoch=10):

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    if model.regularization == "weight_decay":
        optimizer = make_optimizer(optimizer_name, model.parameters(), lr, weight_decay=model.reg_param)
    else:
        optimizer = make_optimizer(optimizer_name, model.parameters(), lr)

    scheduler = make_scheduler(scheduler_name, optimizer)

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    confusion_matrices = []
    best_val_loss = float('inf')
    epoch_counter = 0

    for epoch in range(epochs):
        
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * xb.size(0)

            if model.regularization == "l1":
                l1 = 0
                for p in model.parameters():
                    l1 = l1 + p.abs().sum()
                loss = loss + model.reg_param * l1

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            preds = out.argmax(1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

        train_losses.append(total_loss / total)
        train_accs.append(correct / total)

        
       
        model.eval()
        total_loss, correct, total = 0.0, 0, 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                loss = criterion(out, yb)
                total_loss += loss.item() * xb.size(0)

                preds = out.argmax(1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(yb.cpu().numpy())

        val_loss = total_loss / total
        val_losses.append(val_loss)
        val_accs.append(correct / total)
        confusion_matrices.append(confusion_matrix(all_labels, all_preds, labels=list(range(model.num_classes))))

        if scheduler is not None:
            scheduler.step()
            
        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_losses[-1]:.4f}, '
              f'Val Loss: {val_loss:.4f}, Val Acc: {val_accs[-1]:.4f}')

        
        if model.regularization == "early_stopping":
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epoch_counter = 0
            else:
                epoch_counter += 1
            if epoch_counter >= limit_epoch:
                print(f"Early stopping at epoch {epoch+1}")
                break

    return {
        "loss_train": train_losses,
        "accuracy_train": train_accs,
        "loss_val": val_losses,
        "accuracy_val": val_accs,
        "confusion_matrices": confusion_matrices,
        "model": model,
    }

Running Function to call with good parametters

In [11]:
def run_training(train_dataset, val_dataset, num_classes, hidden_sizes=(64,), activation_function="relu", regularization=None, reg_param=0.0,
                 optimizer_name="SGD", scheduler_name=None, epochs=40, lr=0.001, batch_size=64, limit_epoch=10, seed=42, convolution=None, in_channels=None, out_channels=None, kernel_size=None, pooling=None):
    np.random.seed(seed)
    torch.manual_seed(seed)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    first_image, first_label = train_dataset[0]
    
    if convolution is not None:
        convol = convolution(in_channels, out_channels, kernel_size, pooling)
        input_size = convol(first_image.unsqueeze(0)).numel()
    else:
        convol = None
        input_size = first_image.numel()

    net = Net(input_size, hidden_sizes=hidden_sizes, num_classes=num_classes, convolution = convol, activation_function=activation_function, regularization=regularization,
              reg_param=reg_param)

    metrics = train(net, train_loader, val_loader, epochs=epochs, lr=lr, optimizer_name=optimizer_name, scheduler_name=scheduler_name, limit_epoch=limit_epoch)

    return metrics, net

Temporary random tests

In [12]:
def plot_metrics(metrics, cm_epochs=()):
    epochs = range(1, len(metrics["loss_train"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, metrics["accuracy_train"], label="Train")
    axes[0].plot(epochs, metrics["accuracy_val"], label="Validation")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(epochs, metrics["loss_train"], label="Train")
    axes[1].plot(epochs, metrics["loss_val"], label="Validation")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    for epoch in cm_epochs:
        index = epoch - 1
        if index < 0 or index >= len(metrics["confusion_matrices"]):
            print("No confusion matrix for epoch", epoch)
            continue

        cm = metrics["confusion_matrices"][index]
        plt.figure(figsize=(8, 8))
        plt.imshow(cm, cmap="Blues")
        plt.colorbar()
        plt.title("Confusion matrix - epoch " + str(epoch))
        plt.xlabel("Predicted class")
        plt.ylabel("True class")
        plt.show()

We also added a function to determine the number of parameters

In [13]:
def nb_parameters(model):
    n_params = 0
    for p in model.parameters():
        n_params += p.numel()
    
    return n_params

Let's start our process with default parameters. For the activation function we will use "relu". We will not use any regularizations, not any schedulers and for the optimizer we will use SGD by default for now.

For the convolution, we use first no convolution at all even we now this is a bad idea. It mean we will just flat the images and give it to the model.

Let's arbitrarily set the learning rate to 0.001 and the number of epochs to 20 for now. Our first objective is to find the best hiddens layers configurations to use it for the rest of the process.
Let's first work with the set of 20 sport (with somes differents and some similar)

Note : In the idea, our results should be the worst at the beginning of the process and the best at the end.

In [14]:
metrics_Gray_12c_256h128h, net_Gray_12c_256h128h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[256, 128], epochs=20, lr=0.001)
# plot_metrics(metrics_Gray_12c_256h128h, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_256h128h)))

Epoch [1/20], Train Loss: 2.7708, Val Loss: 2.5951, Val Acc: 0.0833
Epoch [2/20], Train Loss: 2.4672, Val Loss: 2.5480, Val Acc: 0.1000
Epoch [3/20], Train Loss: 2.4002, Val Loss: 2.4969, Val Acc: 0.2000
Epoch [4/20], Train Loss: 2.3336, Val Loss: 2.3800, Val Acc: 0.2333
Epoch [5/20], Train Loss: 2.2522, Val Loss: 2.3196, Val Acc: 0.2000
Epoch [6/20], Train Loss: 2.2158, Val Loss: 2.3018, Val Acc: 0.2000
Epoch [7/20], Train Loss: 2.1698, Val Loss: 2.3171, Val Acc: 0.1500
Epoch [8/20], Train Loss: 2.1443, Val Loss: 2.2634, Val Acc: 0.1833
Epoch [9/20], Train Loss: 2.1011, Val Loss: 2.2034, Val Acc: 0.2833
Epoch [10/20], Train Loss: 2.0556, Val Loss: 2.2631, Val Acc: 0.2500
Epoch [11/20], Train Loss: 2.0184, Val Loss: 2.2492, Val Acc: 0.1833


KeyboardInterrupt: 

In [ ]:
metrics_Gray_12c_128h64h, net_Gray_12c_128h64h= run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001)
# plot_metrics(metrics_Gray_12c_128h64h, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_128h64h)))

In [ ]:
metrics_Gray_12c_128h, net_Gray_12c_128h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128], epochs=20, lr=0.001)
# plot_metrics(metrics_Gray_12c_128h, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_128h)))

In [ ]:
metrics_Gray_12c_512h_128h, net_Gray_12c_512h_128h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[512, 128], epochs=20, lr=1e-3)
# plot_metrics(metrics_Gray_12c_512h_128h, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_512h_128h)))

In [ ]:
metrics_Gray_12c_64h_32h, net_Gray_12c_64h_32h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[64, 32], epochs=20, lr=1e-3)
# plot_metrics(metrics_Gray_12c_64h_32h, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_64h_32h)))

In [ ]:
metrics_Gray_12c_128h_64h_32h, net_Gray_12c_128h_64h_32h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64, 32], epochs=20, lr=1e-3)
# plot_metrics(metrics_Gray_12c_128h_64h_32h, [10, 20])
print("\nNumber of parameters :" + str(nb_parameters(net_Gray_12c_128h_64h_32h)))

Comparing differents results and not focucing on overfitting of ran, best results are gets with x hiddens layers of 256 and 128 neurons which represent 12 880 788 parameters. But we choosed to use 128 and 64 which is a bit less efficient but represent the half of parameters. It will be easier then, so we will use it for the next of the process.

Let's add now a 2D convolution. As we don't know what is the best configuration (well of course we have theorical ideas but we need to check in practice), we will try with differents kernel sizes (3 and 7). We also need to determine the number of output channel of the convolution which is the number of filter of the convolution. We will try 3 arbitral values : 8, 16 and 32. It represent 6 ran.
Let's do it

In [ ]:
metrics_Gray_12c_8o_3k_poolmax, net_Gray_12c_8o_3k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
# plot_metrics(metrics_Gray_12c_8o_3k_poolmax, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_8o_3k_poolmax)))

In [ ]:
metrics_Gray_12c_16o_3k_poolmax, net_Gray_12c_16o_3k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=16, kernel_size=3, pooling="max")
# plot_metrics(metrics_Gray_12c_16o_3k_poolmax, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_16o_3k_poolmax)))

In [ ]:
metrics_Gray_12c_32o_3k_poolmax, net_Gray_12c_32o_3k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=32, kernel_size=3, pooling="max")
# plot_metrics(metrics_Gray_12c_32o_3k_poolmax, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_32o_3k_poolmax)))

In [ ]:
metrics_Gray_12c_8o_7k_poolmax, net_Gray_12c_8o_7k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=7, pooling="max")
# plot_metrics(metrics_Gray_12c_8o_7k_poolmax, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_8o_7k_poolmax)))

In [ ]:
metrics_Gray_12c_16o_7k_poolmax, net_Gray_12c_16o_7k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=16, kernel_size=7, pooling="max")
# plot_metrics(metrics_Gray_12c_16o_7k_poolmax, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_16o_7k_poolmax)))

In [ ]:
metrics_Gray_12c_32o_7k_poolmax, net_Gray_12c_32o_7k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=32, kernel_size=7, pooling="max")
# plot_metrics(metrics_Gray_12c_32o_7k_poolmax, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_32o_7k_poolmax)))

Best results are get with kernel dimmensions of 3. And the more filter we have, the more accurate is the model. But since the number of parameters explode with the number of filter, we'll keep working with 8 filters.

Let's compare it with average pooling. 

In [ ]:
metrics_Gray_12c_8o_3k_poolaverage, net_Gray_12c_8o_3k_poolaverage = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="average")
# plot_metrics(metrics_Gray_12c_8o_3k_poolaverage, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_32o_3k_poolaverage)))

As expected, poolmax is better than poolaverage.

Until here, we used relu because we know by theoritical that this is the best but we should compare the results between the two others activations functions

In [ ]:
metrics_Gray_12_tanh, net_Gray_12c_tanh = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], activation_function="tanh", epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
# plot_metrics(metrics_Gray_12_tanh, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_tanh)))

In [ ]:
metrics_Gray_12c_sigmoid, net_Gray_12c_sigmoid = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], activation_function="sigmoid", epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
# plot_metrics(metrics_Gray_12c_sigmoid, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_sigmoid)))

Here as we see, relu still be the best option. Now, let's focus on the loss computation to check model behaviour

In [ ]:
metrics_Gray_12c, net_Gray_12c = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Gray_12c, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c)))

We see that the model is overfitting. We then need to add some regularization to prevent our modele to to so. Let's try with weight decay and 0.0001 penalty 

In [ ]:
metrics_Gray_12c_weight_decay, net_Gray_12c_weight_decay = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.0001, epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Gray_12c_weight_decay, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_weight_decay)))

In [ ]:
metrics_Gray_12c_weight_decay, net_Gray_12c_weight_decay = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.001, epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Gray_12c_weight_decay, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_weight_decay)))

In [ ]:
metrics_Gray_12c_weight_decay, net_Gray_12c_weight_decay = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.01, epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Gray_12c_weight_decay, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_weight_decay)))

In [ ]:
metrics_Gray_12c_weight_decay, net_Gray_12c_weight_decay = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.1, epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Gray_12c_weight_decay, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_weight_decay)))

Overfitting is now fixed. Now, let's run our model with color. In theory, accurate results should be better. We also want to check if the model do not overfit

In [ ]:
metrics_Color_12c, net_Color_12c = run_training(train_ds_Color_12, val_ds_Color_12, num_classes=20, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.1, epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=3, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Color_12c, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Color_12c)))

We also should also add better optimizer than SGD to get a better weight computation and schedulers to have the best possible learning rate.

In [ ]:
metrics_Color_12c, net_Color_12c = run_training(train_ds_Color_12, val_ds_Color_12, num_classes=20, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.1, optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=3, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Color_12c, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Color_12c)))

Well we found parameters that seems to be relevants.

What come then is just notes

In [ ]:
metrics = run_training(train_dsGray_100, val_dsGray_100, num_classes=100, hidden_sizes=(256, 128), epochs=10, lr=0.001, convolution=Conv_MaxPool,
                       in_channels=1, out_channels=10, kernel_size=5, pooling="max")
plot_metrics(metrics, [1, 6, 10])

In [ ]:
metrics = run_training(train_dsGray_100, val_dsGray_100, num_classes=100, hidden_sizes=(256, 128), epochs=10, lr=0.001)
plot_metrics(metrics, [1, 7, 12])

note pour plus tard : For the convolution, we use first one single 2d convolution 

In [ ]:
metrics = run_training(train_dsColor_6, val_dsColor6, num_classes=6, hidden_sizes=(256, 128), epochs=10, lr=1e-3)
plot_metrics(metrics, [1, 6, 10])

In [ ]:
metrics = run_training(train_dsColor_6, val_dsColor_6, num_classes=6, hidden_sizes=(256, 128), epochs=10, lr=1e-3, convolution=Conv_MaxPool,
                       in_channels=3, out_channels=10, kernel_size=5)
plot_metrics(metrics, [1, 6, 10])

In [ ]:
metrics = run_training(train_datasetColor6, val_datasetColor_6, num_classes=6, hidden_sizes=(128, 64), epochs=10, lr=1e-3, convolution=Conv_MaxPool,
                       in_channels=3, out_channels=10, kernel_size=5)
plot_metrics(metrics, [1, 6, 10])

In [ ]:
metrics = run_training(train_dsColor_6, val_dsColor_6, num_classes=6, hidden_sizes=(128, 64), epochs=10, lr=1e-3, convolution=Conv_MaxPool,
                       in_channels=3, out_channels=10, kernel_size=5)
plot_metrics(metrics, [1, 6, 10])

In [ ]:
metrics = run_training(train_dsColor_6, val_dsColor_6, num_classes=6, hidden_sizes=(128, 64), epochs=20, lr=1e-3, convolution=Conv_MaxPool,
                       in_channels=3, out_channels=20, kernel_size=5)
plot_metrics(metrics, [1, 6, 10])